# The Unified Credit Risk Engine: IFRS 9 & Basel

**Objective:** This notebook bridges the gap between **Accounting** (IFRS 9 Expected Credit Loss) and **Regulatory Solvency** (Basel Unexpected Loss). 

It demonstrates the complete mathematical lifecycle of a loan portfolio:
1. **EDA:** Simulating a realistic Lognormal PD and EAD distribution.
2. **IFRS 9:** Calculating Expected Loss (EL) using Survival Analysis and Staging.
3. **Basel III:** De-cycling to Through-the-Cycle (TTC) PD and calculating Capital.
4. **Validation:** Proving the Basel Asymptotic formula via Monte Carlo simulation.


## 1. Environment Setup
We use `numpy`, `pandas`, `scipy.stats` (for the normal distribution math), and `matplotlib/seaborn` for visualizations. All output tables will be saved to the `outputs/summaries/` directory.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
import dataclasses
from dataclasses import dataclass, fields
from pathlib import Path

# Setup directories for saving outputs
BASE = Path(r"C:\CreditRiskCourse\01_microcredit_portfolio")
SUM = BASE / "outputs/summaries"
FIG = BASE / "outputs/figures"
SUM.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

# Plot styling
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
print("Environment ready. Outputs will be saved to:", SUM)

## 2. Exploratory Data Analysis (The Loan Book)

**The Lognormal Distribution:** In credit risk, both PDs and Loan Balances (EAD) are assumed to be Lognormal. Why? A normal distribution would allow for negative PDs or negative balances, which is impossible. The lognormal curve has a "long right tail"—most loans are safe and small, but a few are highly toxic or massive, which matches real-world portfolios.


In [ ]:
rng = np.random.default_rng(42)
N_LOANS = 15000 

book = pd.DataFrame({
    "loan_id": np.arange(N_LOANS),
    "ead": np.clip(rng.lognormal(np.log(50000), 0.8, N_LOANS), 5000, 500000).round(0),
    "eir": rng.uniform(0.12, 0.30, N_LOANS).round(4),
    "remaining_months": rng.choice([6, 12, 24, 36, 48, 60], N_LOANS),
    "pd12_orig": np.clip(rng.lognormal(np.log(0.05), 0.6, N_LOANS), 0.002, 0.50),
    "dpd": rng.choice([0, 15, 35, 65, 95, 120], N_LOANS, p=[.80, .07, .05, .03, .03, .02]),
})

book["pd12_now"] = np.clip(book.pd12_orig * rng.lognormal(0, 0.5, N_LOANS), 0.002, 0.95)

# Plot PD and EAD Side-by-Side
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: PD Distribution
sns.histplot(book["pd12_orig"], bins=60, kde=True, color="purple", stat="density", ax=axes[0])
axes[0].set_title("Distribution of Origination PDs")
axes[0].set_xlabel("12-Month Probability of Default")
axes[0].set_ylabel("Density")
axes[0].axvline(book["pd12_orig"].mean(), color='red', linestyle='--', label=f"Mean PD ({book['pd12_orig'].mean():.2%})")
axes[0].legend()

# Plot 2: EAD Distribution
sns.histplot(book["ead"], bins=60, kde=True, color="teal", stat="density", ax=axes[1])
axes[1].set_title("Distribution of Loan Balances (EAD)")
axes[1].set_xlabel("Exposure at Default ($)")
axes[1].set_ylabel("Density")
axes[1].axvline(book["ead"].mean(), color='red', linestyle='--', label=f"Mean EAD (${book['ead'].mean():,.0f})")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG / "lognormal_eda.png", dpi=120, bbox_inches="tight")
plt.show()

## 3. The IFRS 9 Engine: Policy & Staging

**Governance:** The engine will not run until the Credit Committee sets the policy. We use a `Policy` dataclass. 
**Staging (SICR):** A loan moves to Stage 2 (Lifetime ECL) if its PD doubles (relative), rises by 5% (absolute), or goes 30+ days past due.


In [ ]:
@dataclass
class Policy:
    sicr_relative_pd_multiple: float = None
    sicr_absolute_pd_increase: float = None
    dpd_backstop_stage2: int = None
    default_dpd: int = None
    asset_correlation: float = None
    lgd: float = None
    scenario_weights: dict = None
    scenario_z: dict = None

    def outstanding(self):
        return [f.name for f in fields(self) if getattr(self, f.name) is None]

    def require_complete(self):
        todo = self.outstanding()
        if todo: raise ValueError(f"❌ Policy decisions outstanding: {', '.join(todo)}")

POLICY = Policy(
    sicr_relative_pd_multiple=2.0, 
    sicr_absolute_pd_increase=0.05,
    dpd_backstop_stage2=30, 
    default_dpd=90,
    asset_correlation=0.15, 
    lgd=0.45,
    scenario_weights={"base": 0.5, "upside": 0.2, "downside": 0.3},
    scenario_z={"base": [0, 0, 0, 0, 0],
                "upside": [0.5, 0.5, 0.25, 0, 0],
                "downside": [-1.5, -1.0, -0.5, 0, 0]}
)

def assign_stage(df, pol):
    stage2 = ((df.pd12_now / df.pd12_orig >= pol.sicr_relative_pd_multiple) | 
              (df.pd12_now - df.pd12_orig >= pol.sicr_absolute_pd_increase) | 
              (df.dpd >= pol.dpd_backstop_stage2))
    stage = np.where(stage2, 2, 1)
    return np.where(df.dpd >= pol.default_dpd, 3, stage)

book["stage"] = assign_stage(book, POLICY)
print(book["stage"].value_counts().sort_index().to_string())

## 4. Survival Analysis & IFRS 9 Expected Loss (EL)

A 5-year loan doesn't have a flat default risk. The risk of defaulting in Year 3 is conditional on surviving Years 1 and 2. We convert the annual PD into a **monthly hazard rate** to calculate the Marginal PD for each month.


In [ ]:
def pit_adjust(pd_ttc, z, rho):
    return norm.cdf((norm.ppf(pd_ttc) - np.sqrt(rho) * z) / np.sqrt(1 - rho))

def marginal_pd_matrix(pd12, z_path, rho, T):
    months = np.arange(1, T + 1)
    z_m = np.asarray(z_path, float)[np.minimum((months - 1) // 12, len(z_path) - 1)] 
    pd_year = pit_adjust(pd12[:, None], z_m[None, :], rho) 
    h = 1 - (1 - pd_year) ** (1 / 12) 
    surv = np.cumprod(1 - h, axis=1)
    surv_prev = np.hstack([np.ones((len(pd12), 1)), surv[:, :-1]])
    return surv_prev * h 

def ead_matrix(balance, eir, n_rem, T):
    r = (eir / 12)[:, None]
    pmt = (balance[:, None] * r) / (1 - (1 + r) ** (-n_rem[:, None]))
    k = np.arange(0, T)[None, :]
    bal = balance[:, None] * (1 + r) ** k - pmt * ((1 + r) ** k - 1) / r
    alive = np.arange(1, T + 1)[None, :] <= n_rem[:, None]
    return np.where(alive, np.maximum(bal, 0.0), 0.0)

def ecl_for_scenario(df, pol, scenario, stage, T=60):
    months = np.arange(1, T + 1)
    mpd = marginal_pd_matrix(df.pd12_now.to_numpy(), pol.scenario_z[scenario], pol.asset_correlation, T)
    ead = ead_matrix(df.ead.to_numpy(), df.eir.to_numpy(), df.remaining_months.to_numpy(), T)
    disc = (1 + df.eir.to_numpy()[:, None] / 12) ** (-months[None, :])
    
    horizon = np.where(stage == 1, np.minimum(12, df.remaining_months.to_numpy()), df.remaining_months.to_numpy())
    in_horizon = months[None, :] <= horizon[:, None]
    
    ecl = (mpd * ead * disc * in_horizon).sum(axis=1) * pol.lgd
    return np.where(stage == 3, pol.lgd * df.ead.to_numpy(), ecl)

per = {s: ecl_for_scenario(book, POLICY, s, book.stage) for s in POLICY.scenario_weights}
book["ecl"] = sum(w * per[s] for s, w in POLICY.scenario_weights.items())

summary = book.groupby("stage").agg(loans=("loan_id", "size"), exposure=("ead", "sum"), ecl=("ecl", "sum"))
summary["coverage_%"] = (100 * summary.ecl / summary.exposure).round(2)

summary.to_csv(SUM / "06_ifrs9_staging_summary.csv")
print(summary.round(1))

## 5. The Basel Engine: De-cycling & Capital


In [ ]:
def decycle_to_ttc(pit_pd, z_current, rho):
    z_pit = norm.ppf(pit_pd)
    z_ttc = (z_pit * np.sqrt(1 - rho)) + (z_current * np.sqrt(rho))
    return norm.cdf(z_ttc)

def calculate_basel_capital(ttc_pd, lgd, rho):
    z_999 = norm.ppf(0.999)
    worst_case_pd = norm.cdf((norm.ppf(ttc_pd) + np.sqrt(rho) * z_999) / np.sqrt(1 - rho))
    K = (lgd * worst_case_pd) - (ttc_pd * lgd)
    return worst_case_pd, K

loan = book.iloc[0]
pit_pd = loan.pd12_now
z_today = -1.0
rho = POLICY.asset_correlation
lgd = POLICY.lgd

ttc_pd = decycle_to_ttc(pit_pd, z_today, rho)
wc_pd, K = calculate_basel_capital(ttc_pd, lgd, rho)
capital_dollars = K * loan.ead

print(f"Current PIT PD: {pit_pd:.2%} -> De-cycled TTC PD: {ttc_pd:.2%}")
print(f"Worst-Case PD (99.9%): {wc_pd:.2%}")
print(f"Basel Capital Requirement: ${capital_dollars:,.2f}")

## 6. Monte Carlo Convergence & Granularity

The Basel ASRF formula assumes an infinite number of loans (Asymptotic). We run a Monte Carlo simulation to prove that as the portfolio size grows, the simulation converges to the formula. 

We separate the small portfolios (fat tails) from the large portfolios (convergence).


In [ ]:
def run_granularity_tests():
    PD = 0.04; LGD = 0.45; EAD = 100_000; RHO = 0.15; n_sim = 100_000
    portfolio_sets = {
        "mc_small_portfolios.png": [100, 200, 500, 1_000],
        "mc_large_portfolios.png": [5_000, 20_000, 100_000, 500_000]
    }
    all_results = []

    for filename, sizes in portfolio_sets.items():
        print(f"\nGenerating: {filename} for sizes {sizes}...")
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        axes = axes.flatten()
        
        for i, N in enumerate(sizes):
            TOTAL_EAD = N * EAD
            EL_RATE = PD * LGD
            
            np.random.seed(42) 
            Z = np.random.standard_normal(n_sim)
            pd_pit = norm.cdf((norm.ppf(PD) - np.sqrt(RHO) * Z) / np.sqrt(1 - RHO))
            
            if N > 10_000:
                defaults = np.maximum(0, np.random.normal(N * pd_pit, np.sqrt(N * pd_pit * (1 - pd_pit))))
            else:
                defaults = np.random.binomial(N, pd_pit)
                
            loss_rates = (defaults * EAD * LGD) / TOTAL_EAD
            mc_var = np.percentile(loss_rates, 99.9)
            
            worst_case_pd = norm.cdf((norm.ppf(PD) + np.sqrt(RHO) * norm.ppf(0.999)) / np.sqrt(1 - RHO))
            basel_var = worst_case_pd * LGD
            
            # Store results
            mc_ul_pct = (mc_var - EL_RATE) * 100
            basel_ul_pct = (basel_var - EL_RATE) * 100
            diff_pct = abs(mc_ul_pct - basel_ul_pct) / basel_ul_pct * 100
            all_results.append({"Portfolio Size": N, "MC UL %": mc_ul_pct, "Basel UL %": basel_ul_pct, "Difference %": diff_pct})
            
            # Plotting
            ax = axes[i]
            sns.histplot(loss_rates, bins=100, kde=True, ax=ax, color="steelblue", stat="density", alpha=0.4)
            ax.axvline(EL_RATE, color="blue", linestyle="--", linewidth=2, label="Expected Loss")
            ax.axvline(mc_var, color="red", linewidth=2, label="MC VaR 99.9%")
            ax.axvline(basel_var, color="black", linestyle="-.", linewidth=2, label="Basel VaR 99.9%")
            
            ax.set_title(f"Portfolio: {N:,} Loans", fontsize=12, fontweight='bold')
            ax.set_xlim(0.00, 0.18)
            if i == 0: ax.legend(loc="upper right", fontsize=8)

        plt.tight_layout()
        plt.savefig(FIG / filename, dpi=120, bbox_inches="tight")
        plt.show()

    df_results = pd.DataFrame(all_results)
    df_results.to_csv(SUM / "06_monte_carlo_granularity.csv", index=False)
    print("\n--- PORTFOLIO GRANULARITY TEST ---")
    print(df_results.to_string(index=False, formatters={'MC UL %': '{:.2f}%'.format, 'Basel UL %': '{:.2f}%'.format, 'Difference %': '{:.2f}%'.format}))

run_granularity_tests()

## 7. Sensitivity Analysis (Playing with Committee Decisions)


In [ ]:
# 1. SICR Sensitivity
sicr_results = []
for mult in [1.5, 2.0, 3.0]:
    new_pol = dataclasses.replace(POLICY, sicr_relative_pd_multiple=mult)
    stage = assign_stage(book, new_pol)
    per = {s: ecl_for_scenario(book, new_pol, s, stage) for s in new_pol.scenario_weights}
    ecl = sum(w * per[s] for s, w in new_pol.scenario_weights.items())
    sicr_results.append({"SICR Threshold": f"{mult}x", "Stage 2 Share %": (stage==2).mean()*100, "Total ECL": ecl.sum()})

df_sicr = pd.DataFrame(sicr_results)
df_sicr.to_csv(SUM / "06_sensitivity_sicr.csv", index=False)
print("--- SENSITIVITY: SICR Relative PD Multiple ---")
print(df_sicr.to_string(index=False, formatters={'Stage 2 Share %': '{:.1f}%'.format, 'Total ECL': '${:,.0f}'.format}))

# 2. Scenario Weight Sensitivity
scen_results = []
for w in [0.1, 0.3, 0.5]:
    new_weights = {"base": 1-w-0.1, "upside": 0.1, "downside": w}
    new_pol = dataclasses.replace(POLICY, scenario_weights=new_weights)
    per = {s: ecl_for_scenario(book, new_pol, s, book.stage) for s in new_pol.scenario_weights}
    ecl = sum(w * per[s] for s, w in new_pol.scenario_weights.items())
    scen_results.append({"Downside Weight": f"{w:.0%}", "Total ECL": ecl.sum()})

df_scen = pd.DataFrame(scen_results)
df_scen.to_csv(SUM / "06_sensitivity_scenarios.csv", index=False)
print("\n--- SENSITIVITY: Downside Scenario Weight ---")
print(df_scen.to_string(index=False, formatters={'Total ECL': '${:,.0f}'.format}))